# Calculate 1$^\circ$-based parameters and aggregate to shapefiles
For resolution testing.

For main analysis / figures, all using 1-deg QDM and 1-deg QDM-byyr runs:
- QPLAD `tassumpoly` to impact regions 
- QPLAD `tassumpoly` to ADM1s
- QPLAD `tasmaxbinF` to US counties
- Will need to calculate `tassumpoly`/`tasmaxbinF` for 1-deg QDM / QDM-byyr runs first

Run days90plus and tassumpoly processing in `2_aggregate_to_shps.ipynb` after this as well.

In [ ]:
import xarray as xr
import xagg as xa
import numpy as np
import pandas as pd
import geopandas as gpd
import os
import re
import glob
from numba import jit as njit
import zarr
import warnings
from tqdm.notebook import tqdm
from dask.diagnostics import ProgressBar

import distributed

from funcs_support import get_filepaths,get_params,utility_save
from funcs_aux import _find_main_variable

dir_list = get_params()

In [ ]:
from distributed import Client

# Start dask client 
# (playing around with workers, threads for efficiency)
client = Client(n_workers=6,
    threads_per_worker=3,
    processes=True,
    memory_limit="8GB")
display(client)

In [ ]:
# Get paths of BCD-ME files
bcd_fns = pd.DataFrame({'path':np.sort(glob.glob(dir_list['bcd_me']+'*.zarr'))})
bcd_fns['model'] = [re.split(r'\.',re.split(r'\_',fn)[-1])[0] for fn in bcd_fns.path]
bcd_fns['typ'] = [re.split(r'_',re.split(r'\/',fn)[-1])[2] for fn in bcd_fns.path.values]
bcd_fns = bcd_fns.set_index(['typ','model'])

In [ ]:
params_proc = {'impact-regions':{'varnames':['tassumpoly'],'shp_fn':'impact-region'},
               'adm1':{'varnames':['tassumpoly'],'shp_fn':'country-fixed'},
               'adm2-us':{'varnames':['tasmaxbinF'],'shp_fn':'UScounties'}}


idxs = {'impact-regions':{'hierid':'locid'},
        'adm1':{'CNTRY_NAME':'locid'},
        'adm2-us':{'FIPS':'locid'}}

In [ ]:
# Load population density
pop_fn = dir_list['aux']+'pop_data/gpw_v4_population_count_rev11_025deg_rg.nc'
pop = xr.open_dataset(pop_fn)

In [ ]:
@njit
def numba_expsums(data,degs,C_from_K = False):
    ''' Numbaized sum of exponents function
    '''

    sums = np.full(len(degs),np.nan)

    if C_from_K:
        data = data-273.15

    if not np.all(np.isnan(data)):
        for i,deg in enumerate(degs):
            sums[i] = sum(data**deg)

    return sums
    

def dmgf_params_carleton(da,degs,dim='dayofyear'):
    ''' Calculate necessary parameters for Carleton et al. Damage Function

    Get T, T^2, T^3, T^4 for each location

    Compare to the equation in B.2.4 (Supp Material), which is 
       T_it = [ Sum_day ( Sum_pix ( w_pix (T_pix,day)^k ) ) for k in 1, 2, 3, 4]

    Since summations are commutative, I change this to 
       T_it = [ Sum_pix ( w_pix * Sum _day (T_pix,day)^k ) for k in 1, 2, 3, 4]

    Parameters
    ---------------
    da : :py:meth:`xr.DataArray`

    degs : list / array
        Exponential degrees (the ks in T**k), e.g. [1,2,3,4]
        
    dim : :py:meth:`str`, by default 'dayofyear'
        The dimension over which to conduct the histogram 

    Returns
    ---------------
    params : :py:meth:`xr.DataArray` 
        With the daysofyear dimension replaced with n_params (in this case 4)
    
    '''

    params = xr.apply_ufunc(
        numba_expsums,
        da,
        input_core_dims=[[dim]],
        output_core_dims=[['param']],
        kwargs={'degs': degs},
        vectorize=True,
        dask="parallelized",
        output_dtypes=[np.float32],
        dask_gufunc_kwargs = {'output_sizes':{'param':len(degs)}}
    )
    params['param'] = ['T'+str(k) for k in degs]
    return params


@njit
def numba_histogram(data, bins, edges = 'inf'):
    ''' Numbaized histogram function

    '''
    if edges == 'inf':
        bins = np.array([-np.inf,*bins,np.inf])
        
    out_len = len(bins) - 1

    # int16 more than sufficient for max value of 365
    hist = np.zeros(out_len, dtype=np.int16)
    if not np.all(np.isnan(data)):
        for value in data:
            for i in range(out_len):
                if bins[i] <= value < bins[i + 1]:
                    hist[i] += 1
                    break
    return hist

def dmgf_params_bins(da, bins, dim='dayofyear'):
    ''' Generate histogram bins

    Parameters
    ---------------
    da : :py:meth:`xr.DataArray`

    bins : iterable
        The bins of the histogram

    dim : :py:meth:`str`, by default 'dayofyear'
        The dimension over which to conduct the histogram

    Returns
    ---------------
    bins_out : :py:meth:`xr.DataArray` 
        With the `dim` dimension replaced with n_bins

    '''
    bins_out =  xr.apply_ufunc(
            numba_histogram,
            da,
            input_core_dims=[[dim]],
            output_core_dims=[['bin']],
            kwargs={'bins': bins},
            vectorize=True,
            dask="parallelized",
            output_dtypes=[np.int16],
            dask_gufunc_kwargs = {'output_sizes':{'bin':len(bins)+1}}
        )
    bins_out['bin'] = np.array([bins[0] - np.mean(np.diff(bins))/2,
                               *(bins[0:-1] + np.diff(bins)/2),
                               bins[-1] + np.mean(np.diff(bins))/2])
    
    return bins_out

## Calculate regression parameters for 1-degree QDM files 

In [ ]:
overwrite = False
#for src in ['qdm-byyr','qdm']:#,'qdm-byyr']:
for src in ['qdm-byyr','qdm']:
    # Load sample QDM 1-degree file
    ds_grid = xr.open_zarr(bcd_fns.loc['qdm'].iloc[0]['path'],decode_timedelta=False)[['lat','lon']].load()
    # Weird xagg read-only error in xa.auxfuncs.get_bnds, manually setting bounds should sidestep it
    # (lat bounds already included) (this is a problem with the `impact_uncertainty` environment, but not the `bcd_me` one, hmmm...)
    ds_grid['lon_bnds'] = xr.concat([ds_grid.lon-0.5,
               ds_grid.lon+0.5],
              dim='bnds').transpose('lon','bnds')

    modlist = np.unique(bcd_fns.loc[src].index.values)

    #modlist = ['CanESM5']
    
    #for proc in params_proc:
    #    print(proc)
    for proc in ['impact-regions','adm1','adm2-us']:
        #------------------------
        # Setup -----------------
        #------------------------
        if 'byyr' in src:
            gwl_string = 'ALLTIMEFRAMES'
            src_src = 'qdm-byyr'
        else:
            gwl_string = 'ALLGWLs'
            src_src = 'qdm'
        # Generate filenames
        output_fns = {mod:{var:{'nowght':f'{dir_list['proc']}aggs/{var}_agg-1deg_{mod}_ALLRUNS_{gwl_string}_{proc}.zarr',
                                'popwght':f'{dir_list['proc']}aggs/{var}_agg-1deg_{mod}_ALLRUNS_{gwl_string}_{proc}_popwght.zarr'}
                       for var in params_proc[proc]['varnames']}
                 for mod in modlist}
        
        # Check for completed existence of all files for a given 
        # shapefile (utility_print adds an empty .done flag after saving)
        if (not overwrite) and (np.all([np.all([np.all([os.path.exists(output_fns[k][v][t]+'/.done') 
                                   for t in output_fns[k][v]]) 
                           for v in output_fns[k]]) 
                   for k in output_fns.keys()])):
            print(f'All files generated for {proc}, skipped.')
            continue
        
        #------------------------
        # Generate weightmaps ---
        #------------------------
        # Load shapefile
        gdf = gpd.read_file(f'{dir_list['aux']}geo_data/{params_proc[proc]['shp_fn']}.shp')
        gdf['geometry'] = gdf.make_valid()
        
        # Keep relevant index columns
        gdf = gdf.loc[:,[*[k for k in idxs[proc]],'geometry']]
        
        # Rename index columns to standard names across analyses
        gdf = gdf.rename(idxs[proc],axis=1)

        single_poly = gdf.union_all()
        ds_grid_tmp = ds_grid.sel(lat=slice(single_poly.bounds[1]-1,single_poly.bounds[3]+1),
               lon=slice(single_poly.bounds[0]-1,single_poly.bounds[2]+1))
        
        # Create weightmap
        with warnings.catch_warnings():
            # Catches a weights bbox warning that doesn't affect this process
            warnings.filterwarnings('ignore')
            wms = {'nowght':xa.pixel_overlaps(ds_grid_tmp,gdf),
                   'popwght':xa.pixel_overlaps(ds_grid_tmp,gdf,weights=pop.pop)}
    
    
        #------------------------
        # Aggregate by model ----
        #------------------------
        for mod in tqdm(modlist):
            for var in params_proc[proc]['varnames']:
                # Create placeholder filename - empty file created to show that this instance
                # is working on this set of files. nc so it shows up in get_filepaths(). Will 
                # be deleted once the processing is complete. 
                placeholder_fn = f'{dir_list['proc']}aggs/{var}tmpproc_agg-1deg_{mod}_ALLRUNS_{gwl_string}_{proc}.nc'
            
                if os.path.exists(placeholder_fn):
                    print('Another instance is working on this aggregation, skipping for now.')
                    continue

                # Check for completed existence of files for a given
                # model / variable combination (utility_print adds an
                # empty .done flag after saving)
                path_exists = {wght:os.path.exists(output_fns[mod][var][wght]+'/.done') for wght in output_fns[mod][var]}
                if (not overwrite) and (np.all([v for k,v in path_exists.items()])):
                    print(f'All files for {mod} {var} aggregated to {proc} already generated, skipped.')
                    continue
                if overwrite:
                    proc_weights = [wght for wght in path_exists]
                else:
                    proc_weights = [wght for wght,flag in path_exists.items() if not flag]

                #----------- Create empty placeholder file -----------
                open(placeholder_fn, 'w').close()
                
                
                # Load QDM 1-deg bias-corrected data
                if re.search(r'^tasmax',var):
                    basevar = 'tasmax'
                else:
                    basevar = 'tas'
                
                ds = xr.open_zarr(bcd_fns.loc[(src_src,mod),'path'],decode_timedelta=False)
                has_data = ds.flag_has_data.load()
                if basevar in ds:
                    ds = ds[[basevar]]
                else:
                    print(f'{basevar} not in {bcd_fns.loc[(src_src,mod),'path']}, skipped.')
                    # Remove placeholder file
                    os.remove(placeholder_fn)
                    continue
                
                # Limit to bounding box of shapefiles (useful mainly for the US counties one)
                ds = ds.sel(lat=slice(single_poly.bounds[1]-1,single_poly.bounds[3]+1),
                       lon=slice(single_poly.bounds[0]-1,single_poly.bounds[2]+1))
                
                ds = ds.sel(proj_base = ['ERA5','MERRA2','JRA-3Q','GMFD'])
                
                # Rechunk to allow geographic processing
                #ds = ds.chunk({'lat':-1,'lon':-1,'proj_base':-1})
                ds = ds.chunk({'dayofyear':-1,'year':-1})
    
                
                #------------------------
                # Calculate params ------
                #------------------------
                # Save temporary file for damage function parameters
                # (to make memory / task graph creation less buggy) 
                fn_tmp_params = f'/glade/derecho/scratch/schwarzwald/impact_uncertainty/tmp/{var}_1deg_{mod}_{src_src}_TMP.zarr'
                
                if overwrite or (not os.path.exists(fn_tmp_params+'/.done')):
                    if (not overwrite) and os.path.exists(fn_tmp_params):
                        # If exist, but is not done, then don't touch
                        print('Another instance is working on '+fn_tmp_params+', skipping...')
                        # Remove placeholder file
                        os.remove(placeholder_fn)
                        continue
                        
                    # Create 
                    if 'sumpoly' in var:
                        # Calculate sums of polynomials, used e.g. by the Carleton et al.
                        # damage function, make sure it's in C if a temperature variable
                        ds_tmp = (dmgf_params_carleton((ds[basevar]-
                                                            ('tas' in basevar)*273.15), 
                                                           np.arange(1,5)).
                                      to_dataset(name=basevar+'sumpoly'))
                        ds_tmp = ds_tmp.mean('year')
                    elif 'binF' in var:
                        binsF = np.arange(0, 130, 5).astype(float) 
                        # Calculate days in bins (converting to F from K)
                        ds_tmp = (dmgf_params_bins(ds[basevar]*(9/5) - 459.67, bins=binsF).
                                    to_dataset(name=basevar+'binF'))
                        ds_tmp = ds_tmp.mean('year')
                        ds_tmp = ds_tmp.chunk({'lat':40,'lon':40})
                    
                    # Ensure has_data is kept; it occasionally gets lost for some reason
                    ds_tmp = ds_tmp.assign_coords({'flag_has_data':has_data})
                    with warnings.catch_warnings():
                        # Zarr metadata warnings, etc.
                        warnings.filterwarnings('ignore')
                        ds_tmp.to_zarr(fn_tmp_params,consolidated=False,mode='w')
                        open(fn_tmp_params+'/.done', 'w').close()
                    print(f'Temp file {fn_tmp_params} saved')
                else:
                    print(f'Temp file {fn_tmp_params} loaded!')
    
                
                #------------------------
                # Aggregate -------------
                #------------------------
                ds_tmp = xr.open_zarr(fn_tmp_params)
                # Chunk for geographic processing
                ds_tmp = ds_tmp.chunk({'idv':1,'lat':-1,'lon':-1})

                # Fix into correct dimension name format
                if 'param' in ds_tmp.sizes:
                    ds_tmp['param'] = [int(k[1]) for k in ds_tmp.param.values]
                    ds_tmp = ds_tmp.rename({'param':'degree'})

                # Workaround for a bug in xa exports with singleton dimensions
                single_idv_fix = False
                if ds_tmp.sizes['idv'] == 1:
                    idv_tmp = ds_tmp[['model','experiment','run']]
                    has_data_tmp = ds_tmp[['flag_has_data']]
                    ds_tmp = ds_tmp.isel(idv=0)
                    single_idv_fix = True 
                    
                # Aggregate
                with warnings.catch_warnings():
                    # Stops inconsistent nan warning from xagg - which is just catching that
                    # some gwls are all-nans for some idvs
                    warnings.filterwarnings('ignore')
                    
                    with xa.set_options(impl='numba',silent=True):
                        aggs = {wght:xa.aggregate(ds_tmp,wms[wght])
                                for wght in proc_weights}
                
                # Move aggregated output to xarray dataset format
                aggs_out = {wght:agg.to_dataset()
                            for wght,agg in aggs.items()}
                        
                # Add experiment / run id back (secondary coordinates don't make it
                # through the xagg workflow) 
                for wght in aggs_out:
                    if single_idv_fix:
                        aggs_out[wght] = xr.merge([aggs['nowght'].to_dataset(),
                                                   idv_tmp,has_data_tmp])
                        aggs_out[wght][_find_main_variable(aggs_out[wght])] = aggs_out[wght][_find_main_variable(aggs_out[wght])].expand_dims({'idv':aggs_out[wght].idv})
                        aggs_out[wght] = aggs_out[wght].transpose(*['poly_idx','proj_base','gwl','idv',
                                                                    *[k for k in aggs_out[wght].sizes if k not in ['poly_idx','proj_base','gwl','idv']]])

                    else:
                        aggs_out[wght] = aggs_out[wght].reset_index('idv').assign_coords({var:ds_tmp[var] for var in ['experiment','run','model','flag_has_data']})
        
                #------------------------
                # Save ----
                #------------------------
                for wght in aggs_out:
                    # Metadata
                    if wght == 'nowght':
                        weightdesc = ''
                    elif wght == 'popwght':
                        weightdesc = ' using pop. weights'
                    aggs_out[wght].attrs = {'SOURCE':'aggregate_to_shps.ipynb',
                                            'DESCRIPTION':f'BCD-ME 1-degree {mod} {var} data, aggregated{weightdesc} to {re.sub(r'\-', ' ',proc)} polygons'}
                
                    # Chunking for zarr saving
                    chunk_dims = ['idv','proj_base']
                    full_dims = [dim for dim in aggs_out[wght][var].sizes if dim not in chunk_dims]
                    aggs_out[wght] = aggs_out[wght].chunk({**{dim:1 for dim in chunk_dims},
                              **{dim:-1 for dim in full_dims}}
                            )

                    # Load coordinates, if not loaded (can otherwise cause write errors)
                    for v in aggs_out[wght].coords:
                        aggs_out[wght][v] = aggs_out[wght][v].load()
                
                    # Output as zarr
                    with warnings.catch_warnings():
                        # Zarr metadata warnings, etc.
                        warnings.filterwarnings('ignore')
                        utility_save(aggs_out[wght].drop_encoding(),
                                     output_fns[mod][var][wght],
                                     zarr_mode = 'w',
                                     save_kwargs = {'zarr_format':3})
                    
                # Remove temporary file
                #os.system('rm -rf '+fn_tmp_params)
                #print('Temporary file '+fn_tmp_params+' removed!')

                # Remove placeholder file
                os.remove(placeholder_fn)